<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_5/lessons/lesson_51_ci_cd/note_lesson_51_ci_cd_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚙️ Урок 51 — CI/CD (GitHub Actions)

За уроки 37–50 у `news_hub` з'явилися сотні тестів, `mypy --strict`, тести образу, smoke-тест Compose; у проєкті нотаток — тести Django і `check --deploy`. Але все це працює, лише коли хтось не забув запустити. **CI** робить це сам на кожен pull request, а **CD** після злиття в `main` публікує перевірений образ.

GitHub Actions — це YAML-файли в `.github/workflows/` і кілька простих правил: **коли** запускати, **у якому порядку**, **з якими правами**, що робити, коли крок **упав**. Сьогодні розбираємо справжні workflows курсу (`news_hub.yml`, `crispy_notes.yml`) як дані й **запускаємо їхні кроки локально**: ruff, pytest, перевірки збірки, маскування секретів — тими самими командами, що й на раннері GitHub.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | workflow як дані: пастка `on`, jobs і steps, дії та версії, вирази `${{ }}`, змінні й робоча папка | 0.1–0.7 | |
| 1 | коли запускається: `branches`, `paths`, шаблони `*` / `**`, «мертві» шляхи, `concurrency` | 1.1–1.5 | **Задача 1** |
| 2 | кроки локально: виконавець кроків і job, `set -e`, `\|\| true`, ruff + pytest на міні-проєкті, перевірки проєкту й образу, мінімальні версії, Docker | 2.1–2.10 | |
| 3 | порядок jobs: `needs`, хвилі, справжній розклад, `if:`, що буде, коли job упав, `fail-fast` | 3.1–3.5 | **Задача 2** |
| 4 | matrix: розгортання, `include` / `exclude`, імена jobs | 4.1–4.3 | |
| 5 | кеш: `hashFiles`, ключ кешу pip, збереження | 5.1–5.3 | **Задача 3** |
| 6 | права `GITHUB_TOKEN`, маскування секретів, `$GITHUB_ENV`, секрети в PR з форку | 6.1–6.4 | |
| 7 | знайди помилку: лінтер workflow | 7.1 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, константи, YAML-рядки) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Блоки «Справжній вивід» — журнали прогону на GitHub з уроку; наші моделі мають давати той самий результат.
- Повний розбір — у книзі: [Урок 51](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/lesson_51/).

---
## 0. Workflow як дані

Workflows лежать у **корені** репозиторію (`.github/workflows/`), а проєкти — у папці уроку. Клітинка нижче знаходить корінь курсу вгору від ноутбука; у Colab завантажує з GitHub лише ці дві папки (`sparse-checkout`). Потрібні PyYAML, ruff і pytest — чого немає, вона встановить.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

LESSON = "module_5/lessons/lesson_51_ci_cd"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"


def find_root(start: Path) -> Path | None:
    for folder in (start, *start.parents):
        if (folder / ".github/workflows").is_dir() and (folder / LESSON).is_dir():
            return folder
    return None


ROOT = find_root(Path.cwd())
if ROOT is None:
    if not Path("course_repo").is_dir():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", ".github/workflows", LESSON], check=True)
    ROOT = Path("course_repo").resolve()
os.chdir(ROOT)

for module, package in (("yaml", "pyyaml"), ("ruff", "ruff"), ("pytest", "pytest")):
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import yaml

print("корінь курсу:", ROOT.name, "| workflows:", sorted(p.name for p in Path(".github/workflows").glob("*.yml")))

### Приклад 0.1. Прочитати workflows

In [ ]:
def load_workflows(folder: str = ".github/workflows") -> dict[str, dict]:
    """{ім'я файлу без .yml: workflow як словник}."""
    return {path.stem: yaml.safe_load(path.read_text(encoding="utf-8")) for path in sorted(Path(folder).glob("*.yml"))}

In [ ]:
WORKFLOWS = load_workflows()
NEWS, NOTES = WORKFLOWS["news_hub"], WORKFLOWS["crispy_notes"]
for name, workflow in WORKFLOWS.items():
    print(f"{name:<13} name={workflow['name']!r:<45} jobs={list(workflow['jobs'])}")
print("ключі верхнього рівня news_hub.yml:", list(NEWS))

🔮 **Прогноз:** серед ключів верхнього рівня немає `"on"`, зате є `True`. Чому?

<details>
<summary>Відповідь</summary>

PyYAML читає YAML 1.1, де `on`, `yes`, `off` — булеві значення. Ключ `on:` став `True`. GitHub читає цей файл інакше (для нього `on` — рядок), але будь-який скрипт на PyYAML, що аналізує workflows, має це враховувати. Помічник `triggers()` нижче бере обидва варіанти.
</details>

### Приклад 0.2. Тригери

In [ ]:
def triggers(workflow: dict) -> dict[str, dict]:
    """Блок `on:` — під ключем "on" або True (YAML 1.1); завжди словник {подія: правило}."""
    block = workflow.get("on", workflow.get(True))
    if isinstance(block, str):
        block = {block: {}}
    elif isinstance(block, list):
        block = {event: {} for event in block}
    return {event: (rule or {}) for event, rule in block.items()}

In [ ]:
for name, workflow in WORKFLOWS.items():
    print(f"{name:<13} {list(triggers(workflow))}")
print("pull_request news_hub:", triggers(NEWS)["pull_request"])
print("рядок і список:", triggers({"on": "push"}), triggers({True: ["push", "pull_request"]}))

`on:` буває рядком (`on: push`), списком (`on: [push, pull_request]`) або словником з фільтрами — функція зводить усе до словника. У `news_hub` три події: `push`, `pull_request` і `workflow_dispatch` (кнопка «Run workflow» у вкладці Actions).

### Приклад 0.3. Анатомія: jobs

**Job** — набір кроків на **окремій чистій віртуальній машині** (runner). Jobs без `needs` стартують паралельно.

```
подія (push / pull_request) ──▶ on: гілки, paths — запускати? ──▶ jobs
   job = чиста VM (runs-on)        services: PostgreSQL, Redis поруч
     steps по черзі: uses (готова дія) або run (команда)
       код виходу ≠ 0 ──▶ job червоний, наступні кроки не йдуть
```

In [ ]:
def job_summary(job: dict) -> dict:
    strategy = job.get("strategy", {})
    return {
        "runs-on": job.get("runs-on"),
        "needs": job.get("needs", "—"),
        "matrix": {k: v for k, v in strategy.get("matrix", {}).items()} or "—",
        "services": list(job.get("services", {})) or "—",
        "кроків": len(job.get("steps", [])),
        "timeout": job.get("timeout-minutes"),
    }

In [ ]:
for name, job in NEWS["jobs"].items():
    info = job_summary(job)
    print(f"{name:<13} needs={info['needs']!s:<46} matrix={info['matrix']!s:<28} services={info['services']!s:<21} кроків={info['кроків']}")

Шість jobs. Лише `tests` має matrix (дві версії Python), лише `postgres` — services (PostgreSQL і Redis поруч з job), лише `docker` і `publish` мають `needs`. Кожен — з `timeout-minutes`: завислий тест не їстиме хвилини раннера годинами (типовий ліміт — 6 годин).

### Приклад 0.4. Кроки: `uses` чи `run`

In [ ]:
def step_kind(step: dict) -> tuple[str, str]:
    """('uses', 'дія@версія') або ('run', перший рядок команди)."""
    if "uses" in step:
        return "uses", step["uses"]
    return "run", step["run"].strip().splitlines()[0]

In [ ]:
for job_name in ("tests", "docker"):
    print(f"--- {job_name}")
    for step in NEWS["jobs"][job_name]["steps"]:
        kind, what = step_kind(step)
        print(f"  {kind:<4} {what:<45} {step.get('name', '')}")

`uses` — готова дія з Marketplace (`actions/checkout` бере код репозиторію, `actions/setup-python` ставить Python і кеш pip). `run` — команда shell на раннері. Кроки без `name` GitHub показує за командою — тому довгим крокам у курсі дано зрозумілі імена.

### Приклад 0.5. Які дії й версії використовуються

In [ ]:
from collections import Counter


def actions_used(workflows: dict[str, dict]) -> Counter:
    """Скільки разів кожна дія@версія трапляється в усіх workflows."""
    used = Counter()
    for workflow in workflows.values():
        for job in workflow["jobs"].values():
            for step in job.get("steps", []):
                if "uses" in step:
                    used[step["uses"]] += 1
    return used

In [ ]:
for action, count in actions_used(WORKFLOWS).most_common():
    print(f"{count:>3} × {action}")

Версія після `@` — це тег (`v5`) дії. Тег власник дії може пересунути на інший коміт; найсуворіший варіант — повний SHA коміту (`actions/checkout@08c6903…`), його не змінить ніхто. Курс тримає теги основних версій і оновлює їх свідомо.

### Приклад 0.6. Вирази `${{ … }}`

Рядки з `${{ … }}` GitHub підставляє перед запуском: `matrix.python`, `env.PROJECT`, `github.ref`, `secrets.ІМ'Я`. Маленький «рендерер» для простих виразів-шляхів:

In [ ]:
import re

EXPR = re.compile(r"\$\{\{\s*([A-Za-z_][\w.\-]*)\s*\}\}")


def lookup(context: dict, path: str):
    """'matrix.python' → context['matrix']['python']; чого немає — порожній рядок, як у GitHub."""
    value = context
    for part in path.split("."):
        if not isinstance(value, dict) or part not in value:
            return ""
        value = value[part]
    return value


def render(text: str, context: dict) -> str:
    return EXPR.sub(lambda match: str(lookup(context, match[1])), text)

In [ ]:
pr_context = {"github": {"workflow": NEWS["name"], "ref": "refs/pull/57/merge", "event_name": "pull_request"},
              "env": NEWS["env"], "secrets": {}}
for python in NEWS["jobs"]["tests"]["strategy"]["matrix"]["python"]:
    print("назва job:", render(NEWS["jobs"]["tests"]["name"], {**pr_context, "matrix": {"python": python}}))
print("concurrency:", render(NEWS["concurrency"]["group"], pr_context))
print("кеш pip за файлами:", render(NEWS["jobs"]["tests"]["steps"][1]["with"]["cache-dependency-path"], pr_context))
print("секрет, якого немає:", repr(render("${{ secrets.GEMINI_API_KEY }}", pr_context)))

Одна назва `pytest · Python ${{ matrix.python }}` дала два jobs з різними іменами. Група `concurrency` для PR №57 — `news_hub — CI/CD-refs/pull/57/merge`. Секрет, якого немає, — **порожній рядок**, не помилка: про це — розділ 6.

### Приклад 0.7. Змінні й робоча папка кроку

Змінні середовища кроку складаються з трьох рівнів: `env` workflow → `env` job → `env` кроку (нижчий перекриває вищий). Робоча папка — `working-directory` кроку, інакше `defaults.run` job, інакше `defaults.run` workflow.

In [ ]:
def step_environment(workflow: dict, job_name: str, step: dict) -> tuple[dict, str]:
    job = workflow["jobs"][job_name]
    env = {**workflow.get("env", {}), **job.get("env", {}), **step.get("env", {})}
    folder = (step.get("working-directory")
              or job.get("defaults", {}).get("run", {}).get("working-directory")
              or workflow.get("defaults", {}).get("run", {}).get("working-directory", "."))
    return env, folder

In [ ]:
pytest_step = NEWS["jobs"]["postgres"]["steps"][-1]
env, folder = step_environment(NEWS, "postgres", pytest_step)
print("крок:", pytest_step["run"], "| папка:", folder)
for key, value in env.items():
    print(f"  {key:<18} {value}")

Крок `pytest -q` у job `postgres` бачить `PROJECT` з workflow і три змінні з job: тести йдуть на справжній PostgreSQL і Redis із `services`. Без `TEST_DATABASE_URL` ті самі тести взяли б SQLite у пам'яті — так працює job `tests`.

---
## 1. Коли workflow запускається

Для `pull_request` і `push` GitHub перевіряє два фільтри:

- `branches` — для `pull_request` це гілка, **у яку** зливають PR (base); для `push` — гілка, куди пушать. Немає фільтра — будь-яка;
- `paths` — хоч один змінений файл має збігтися з хоч одним шаблоном. Немає фільтра — будь-які файли.

У шаблонах `*` — будь-що в межах однієї папки, `**` — будь-яка кількість папок (`**/` може бути й порожнім: `**/*.md` збігається і з `README.md`).

### Приклад 1.1. Шаблони шляхів

In [ ]:
def glob_match(path: str, pattern: str) -> bool:
    """Шаблон шляхів GitHub Actions (спрощено): **/ — нуль чи більше папок, ** — будь-що, * — у межах папки."""
    regex, i = "", 0
    while i < len(pattern):
        if pattern.startswith("**/", i):
            regex, i = regex + "(?:.*/)?", i + 3
        elif pattern.startswith("**", i):
            regex, i = regex + ".*", i + 2
        elif pattern[i] == "*":
            regex, i = regex + "[^/]*", i + 1
        else:
            regex, i = regex + re.escape(pattern[i]), i + 1
    return re.fullmatch(regex, path) is not None

In [ ]:
GLOB_CASES = [
    ("news_hub/api.py", "news_hub/*"),
    ("news_hub/bot/handlers.py", "news_hub/*"),
    ("news_hub/bot/handlers.py", "news_hub/**"),
    ("README.md", "**/*.md"),
    ("docs/modules/m5/lesson_51.md", "**/*.md"),
    ("docs/modules/m5/lesson_51.md", "docs/*.md"),
    ("requirements-dev.txt", "requirements*.txt"),
]
for path, pattern in GLOB_CASES:
    print(f"{path:<32} {pattern:<20} → {glob_match(path, pattern)}")

`news_hub/*` не бачить файлів у вкладених папках — для «будь-якої зміни проєкту» потрібне `news_hub/**`. Саме так записано `paths` у workflows курсу.

### Приклад 1.2. Чи запуститься workflow на цей PR

Змінені файли візьмемо з папки проєкту, яку workflow сам називає в `env.PROJECT`.

In [ ]:
def would_run(workflow: dict, event: str, base: str, changed: list[str]) -> bool:
    """Чи запуститься workflow на подію з базовою гілкою base і зміненими файлами changed."""
    rules = triggers(workflow)
    if event not in rules:
        return False
    rule = rules[event]
    if "branches" in rule and not any(glob_match(base, branch) for branch in rule["branches"]):
        return False
    if "paths" in rule and not any(glob_match(file, pattern) for file in changed for pattern in rule["paths"]):
        return False
    return True


PR_BASE = "claude/wonderful-ride-cinspw"          # PR уроку 51 стоїть на гілці попереднього PR, а не на main
CHANGED = [f"{NEWS['env']['PROJECT']}/scripts/smoke.sh", "docs/modules/m5/lesson_51.md"]

In [ ]:
for name in ("news_hub", "crispy_notes", "notebooks", "docs"):
    print(f"{name:<13} PR → {would_run(WORKFLOWS[name], 'pull_request', PR_BASE, CHANGED)!s:<6}"
          f" push у main → {would_run(WORKFLOWS[name], 'push', 'main', CHANGED)}")
assert would_run(NEWS, "pull_request", PR_BASE, CHANGED)
assert not would_run(NOTES, "pull_request", PR_BASE, CHANGED)            # файлів нотаток не змінено
assert not would_run(NEWS, "push", "claude/m5-production", CHANGED)       # push — лише main

На PR запускається `news_hub` (змінено його `smoke.sh`) і `notebooks` (змінено `docs/`), а `crispy_notes` — ні: його файлів не чіпали. `push` у `main` теж запустить `news_hub` — звідти йде публікація образу; `push` в іншу гілку — ні (інакше кожен коміт PR перевірявся б двічі: на `push` і на `pull_request`).

### Приклад 1.3. Workflow книги: PR, якого «не було»

Тригери з `django-tests.yml` Django-книги (шлях переписано на папку проєкту курсу). Чи запустився б він на PR цього уроку?

🔮 **Прогноз:** що побачить автор PR — червоний хрестик чи щось інше?

In [ ]:
BOOK_WORKFLOW = yaml.safe_load(f"""
on:
  pull_request:
    branches: [ main, master ]
    paths:
      - '{NOTES['env']['PROJECT']}/**'
""")
NOTES_CHANGE = [f"{NOTES['env']['PROJECT']}/requirements.txt"]

In [ ]:
print("книга, PR у гілку попереднього PR:", would_run(BOOK_WORKFLOW, "pull_request", PR_BASE, NOTES_CHANGE))
print("книга, PR у main:                 ", would_run(BOOK_WORKFLOW, "pull_request", "main", NOTES_CHANGE))
print("курс,  PR у гілку попереднього PR:", would_run(NOTES, "pull_request", PR_BASE, NOTES_CHANGE))

<details>
<summary>Відповідь</summary>

Нічого: ні хрестика, ні галочки — перевірки просто немає. А саме в цьому PR змінено нижню межу simplejwt, яку ловить лише job мінімальних версій. **Відсутня перевірка виглядає як успіх** — тому в курсі `pull_request` без `branches`.
</details>

### Приклад 1.4. «Мертві» шляхи

Та сама тиха поломка буває, коли папку проєкту **перейменували**, а `paths` у workflow — ні: шаблон не збігається з жодним файлом, і workflow мовчки перестає запускатися на зміни проєкту. Перевіримо кожен шаблон: чи є в репозиторії хоч один файл під ним.

In [ ]:
def dead_patterns(workflow: dict, root: Path = Path(".")) -> list[str]:
    """Шаблони paths (усіх подій), під які не потрапляє жоден наявний файл чи папка."""
    patterns = {pattern for rule in triggers(workflow).values() for pattern in rule.get("paths", [])}
    return sorted(pattern for pattern in patterns if next(root.glob(pattern), None) is None)


MOVED = yaml.safe_load("""
on:
  pull_request:
    paths: ['module_5/lessons/lesson_99_old_name/news_hub/**', '.github/workflows/news_hub.yml']
""")

In [ ]:
print("перейменована папка:", dead_patterns(MOVED))
for name in ("news_hub", "crispy_notes"):
    project = WORKFLOWS[name]["env"]["PROJECT"]
    exists = Path(project).is_dir()
    print(f"{name:<13} мертві шаблони: {dead_patterns(WORKFLOWS[name])}  | папка PROJECT існує: {exists}")

Для вигаданого workflow перший шаблон «мертвий»: папки `lesson_99_old_name` немає. Для workflows курсу список має бути порожнім, а папка `PROJECT` — існувати. Якщо ти бачиш тут непорожній список чи `False` — це саме така поломка: workflow не запуститься на зміни проєкту, а `working-directory` вкаже в нікуди. Виправлення — один рядок `PROJECT` і шаблони `paths`; перевірити — цією ж клітинкою. Ще одна причина мати `.github/workflows/<файл>.yml` серед `paths`: змінив сам workflow — він запуститься й покаже, чи працює.

### Приклад 1.5. `concurrency`: новий push скасовує старий прогін

```yaml
concurrency:
  group: ${{ github.workflow }}-${{ github.ref }}
  cancel-in-progress: true
```

Прогони з однаковою групою не йдуть одночасно: новий **скасовує** попередній, ще не завершений. Змоделюємо три push у той самий PR за дві хвилини і один push у `main`; кожен прогін триває 180 с.

In [ ]:
def concurrency_outcome(runs: list[tuple[int, str]], duration: int = 180) -> list[str]:
    """runs: [(секунда старту, ref)] → що сталося з кожним прогоном."""
    outcome = []
    for i, (start, ref) in enumerate(runs):
        group = render(NEWS["concurrency"]["group"], {"github": {"workflow": NEWS["name"], "ref": ref}})
        later = [s for s, r in runs[i + 1:] if r == ref and s < start + duration]
        result = f"скасовано на {later[0]} с" if later else f"завершено на {start + duration} с"
        outcome.append(f"t={start:>3} с  {group:<40} {result}")
    return outcome


RUNS = [(0, "refs/pull/57/merge"), (40, "refs/pull/57/merge"), (60, "refs/heads/main"), (110, "refs/pull/57/merge")]

In [ ]:
for line in concurrency_outcome(RUNS):
    print(line)

З трьох прогонів PR №57 до кінця дійшов лише останній — він перевіряє актуальний код, а два застарілі не витрачали хвилини раннерів. Прогін `main` у іншій групі — його push у PR не скасовує.

### Задача 1. `paths-ignore`

Крім `paths` (запускати, якщо змінено хоч щось **із** переліку) буває `paths-ignore` — запускати, **якщо змінено хоч щось поза** переліком. Workflow, що не потрібен для правок документації:

```yaml
on:
  pull_request:
    paths-ignore: ['docs/**', '**/*.md']
```

Напиши `would_run_ignore(rule, changed)` → `True`, якщо хоч один файл зі `changed` **не** збігається з жодним шаблоном `rule["paths-ignore"]`. Якщо в `rule` немає `paths-ignore` — `True`. Використовуй `glob_match`.

In [ ]:
def would_run_ignore(rule: dict, changed: list[str]) -> bool:
    # YOUR CODE HERE
    pass


In [ ]:
DOCS_RULE = {"paths-ignore": ["docs/**", "**/*.md"]}
result = would_run_ignore(DOCS_RULE, ["docs/modules/m5/lesson_51.md"])
assert result is False, f"правка лише документації не має запускати workflow, а вийшло {result}"
assert would_run_ignore(DOCS_RULE, ["README.md", "docs/index.md"]) is False, "**/*.md — і файли в корені"
assert would_run_ignore(DOCS_RULE, ["README.md", f"{LESSON}/news_hub/news_hub/api.py"]) is True, "змінено код — запускати"
assert would_run_ignore({}, ["docs/index.md"]) is True, "без paths-ignore — запускати завжди"
print("✅ Задача 1 розв'язана")

---
## 2. Кроки CI локально

Правило курсу: **усе, що робить CI, можна запустити локально тими самими командами**. Workflow лише вирішує, коли й де. Побудуємо маленький «раннер» і проженемо ним справжні перевірки.

### Приклад 2.1. Один крок

На раннері Ubuntu крок `run:` виконує `bash --noprofile --norc -eo pipefail` (якщо задано `shell: bash`; типово — `bash -e`). Результат кроку — **код виходу** останньої команди: 0 — зелений, інше — червоний.

Щоб команди писались, як у workflow (`python -m pytest`), підкладемо в `PATH` папку з посиланням `python` на інтерпретатор ноутбука.

In [ ]:
import tempfile
import time

SHIM = Path(tempfile.mkdtemp(prefix="l51_bin_"))
(SHIM / "python").unlink(missing_ok=True)
(SHIM / "python").symlink_to(sys.executable)
BASE_ENV = {**{k: v for k, v in os.environ.items() if k not in ("FORCE_COLOR", "CLICOLOR_FORCE")},
            "PATH": f"{SHIM}{os.pathsep}{os.environ['PATH']}", "PYTHONDONTWRITEBYTECODE": "1",
            "NO_COLOR": "1", "PY_COLORS": "0", "TERM": "dumb"}            # журнал без кольорових кодів


def run_step(command: str, cwd: str | Path = ".", env: dict | None = None) -> tuple[int, str, float]:
    """Виконати run-крок як на раннері → (код виходу, вивід, секунди)."""
    started = time.perf_counter()
    proc = subprocess.run(["bash", "--noprofile", "--norc", "-eo", "pipefail", "-c", command],
                          cwd=cwd, env={**BASE_ENV, **(env or {})}, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    return proc.returncode, proc.stdout.strip(), time.perf_counter() - started

In [ ]:
for command in ("python --version", "echo 'тести пройшли'", "test -f немає_такого_файлу", "exit 3"):
    status, output, seconds = run_step(command)
    print(f"{command:<28} → код {status}  {output!r}")

`test -f` для відсутнього файлу повертає 1 без жодного виводу — CI покаже червоний крок і код виходу. `exit 3` — будь-яке ненульове число означає «впав».

### Приклад 2.2. Job: що відбувається після червоного кроку

Кроки job ідуть по черзі. Після першого червоного кроку решта **пропускається** (`skipped`) — крім тих, що мають `if: always()` (завжди) чи `if: failure()` (лише коли щось упало). `continue-on-error: true` дозволяє кроку впасти, не роблячи job червоним.

Наш виконавець job підтримує ще дві речі раннера (розділ 6): рядок `::add-mask::значення` у виводі ховає значення в журналі, а рядки `ІМ'Я=значення`, дописані у файл `$GITHUB_ENV`, стають змінними наступних кроків.

In [ ]:
def run_job(steps: list[dict], cwd: str | Path = ".", env: dict | None = None) -> list[dict]:
    """Кроки по черзі, як на раннері: if, continue-on-error, ::add-mask::, $GITHUB_ENV."""
    env, masks, results, failed = dict(env or {}), set(), [], False
    with tempfile.TemporaryDirectory() as tmp:
        github_env = Path(tmp, "github_env")
        for step in steps:
            condition = step.get("if", "success()")
            if not {"success()": not failed, "always()": True, "failure()": failed}[condition]:
                results.append({"name": step["name"], "status": "skipped", "output": ""})
                continue
            github_env.write_text("")
            status, output, seconds = run_step(step["run"], cwd, {**env, "GITHUB_ENV": str(github_env)})
            lines = []
            for line in output.splitlines():
                if line.startswith("::add-mask::"):
                    masks.add(line.removeprefix("::add-mask::"))
                else:
                    lines.append(line)
            output = "\n".join(lines)
            for secret in masks:
                output = output.replace(secret, "***")
            for line in github_env.read_text().splitlines():
                key, _, value = line.partition("=")
                env[key] = value
            if status == 0:
                verdict = "success"
            elif step.get("continue-on-error"):
                verdict = "failure (continue-on-error)"
            else:
                verdict, failed = "failure", True
            results.append({"name": step["name"], "status": verdict, "code": status, "output": output, "seconds": seconds})
    return results


def show_job(results: list[dict], max_lines: int = 6) -> None:
    marks = {"success": "✓", "skipped": "–", "failure": "✗", "failure (continue-on-error)": "!"}
    for result in results:
        tail = f" (код {result['code']})" if result.get("code") else ""
        print(f"{marks[result['status']]} {result['name']}: {result['status']}{tail}")
        for line in result["output"].splitlines()[:max_lines]:
            print(f"      {line}")
    verdict = "червоний" if any(r["status"] == "failure" for r in results) else "зелений"
    print(f"→ job {verdict}")

In [ ]:
TOY_JOB = [
    {"name": "залежності", "run": "echo 'встановлено'"},
    {"name": "необов'язкова перевірка", "run": "echo 'попередження'; exit 1", "continue-on-error": True},
    {"name": "тести", "run": "echo '1 failed'; exit 1"},
    {"name": "збірка образу", "run": "echo 'не має запуститись'"},
    {"name": "журнали при падінні", "if": "failure()", "run": "echo 'docker compose logs …'"},
    {"name": "прибирання", "if": "always()", "run": "echo 'docker compose down -v'"},
]

In [ ]:
show_job(run_job(TOY_JOB))

- Крок з `continue-on-error` упав (`!`), але job не зупинився.
- `тести` впали — `збірка образу` пропущена: немає сенсу збирати те, що не пройшло тести.
- `if: failure()` і `if: always()` виконались саме тому, що щось упало, — так у CI збирають журнали й прибирають (`smoke.sh` курсу робить те саме через `trap cleanup EXIT`).

### Приклад 2.3. `set -e` у багаторядковому кроці і `|| true`

🔮 **Прогноз:** у кроці два рядки: `pytest` (впав) і `echo готово`. Чи буде крок зеленим? А якщо `pytest || true`?

In [ ]:
MULTILINE = [
    {"name": "без захисту", "run": "false   # тут впав pytest\necho готово"},
    {"name": "|| true", "run": "false || true\necho готово"},
    {"name": "конвеєр з pipefail", "run": "false | tee log.txt"},
]

In [ ]:
work = Path(tempfile.mkdtemp(prefix="l51_shell_"))
for step in MULTILINE:
    show_job(run_job([step], cwd=work))

<details>
<summary>Відповідь</summary>

Без захисту крок червоний, і `готово` не надруковано: з `-e` bash зупиняється на першій невдалій команді. З `|| true` код виходу проковтнуто — крок **зелений** за будь-якого результату тестів. Конвеєр з `pipefail` червоний, хоч `tee` успішний.
</details>

(Кожен варіант тут — окремий job з одного кроку, тому падіння першого не пропускає наступні.) `|| true` біля тестів — найнебезпечніший рядок у CI: зелена галочка, яка нічого не перевіряє. Його шукає лінтер у розділі 7.

### Приклад 2.4. Міні-проєкт для локального CI

Щоб побачити справжні ruff і pytest у дії, створимо в тимчасовій папці маленький проєкт: функцію ціни доставки і тести до неї. Варіант `with_bugs=True` додає невикористаний імпорт і неправильний нічний тариф.

In [ ]:
CALC = """# Ціна доставки — міні-проєкт для локального CI.


def delivery_price(distance_km: float, night: bool = False) -> float:
    price = 40 + 12 * distance_km
    return round(price * ({rate} if night else 1), 2)
"""

TESTS = """import unittest

from calc import delivery_price


class DeliveryPriceTest(unittest.TestCase):
    def test_day(self):
        self.assertEqual(delivery_price(5), 100)

    def test_night(self):
        self.assertEqual(delivery_price(5, night=True), 150)
"""


def make_mini_project(with_bugs: bool = False) -> Path:
    folder = Path(tempfile.mkdtemp(prefix="l51_mini_"))
    source = CALC.format(rate="1.6" if with_bugs else "1.5")
    if with_bugs:
        source = "import os\n" + source
    (folder / "calc.py").write_text(source, encoding="utf-8")
    (folder / "test_calc.py").write_text(TESTS, encoding="utf-8")
    return folder

In [ ]:
clean_project, buggy_project = make_mini_project(), make_mini_project(with_bugs=True)
print(sorted(p.name for p in clean_project.iterdir()))
print((buggy_project / "calc.py").read_text(encoding="utf-8"))

### Приклад 2.5. Job «lint + tests» на міні-проєкті

Порядок кроків — як у курсі: **швидкі першими**. ruff перевіряє код за частки секунди без запуску, тож очевидна помилка видна раніше, ніж тести встигнуть стартувати. `--no-cache` і `-p no:cacheprovider` — щоб не лишати кешів у папці.

In [ ]:
LOCAL_CI = [
    {"name": "ruff check — помилки без запуску коду", "run": "python -m ruff check --no-cache --output-format concise ."},
    {"name": "pytest", "run": "python -m pytest -q -p no:cacheprovider --tb=line"},
    {"name": "журнали при падінні", "if": "failure()", "run": "ls"},
]

In [ ]:
print("=== чистий проєкт")
show_job(run_job(LOCAL_CI, cwd=clean_project))
print("\n=== проєкт з помилками")
show_job(run_job(LOCAL_CI, cwd=buggy_project), max_lines=4)

Чистий проєкт — обидва кроки зелені, `журнали` пропущені. У проєкті з помилками ruff знайшов `F401 'os' imported but unused` і повернув 1 — **pytest навіть не запускався**, а неправильний тариф ще не виявлено. Це нормальна поведінка CI: виправ першу причину — побачиш наступну.

Те саме з `continue-on-error` на ruff — тепер видно обидві проблеми за один прогін:

In [ ]:
LINT_SOFT = [{**LOCAL_CI[0], "continue-on-error": True}, *LOCAL_CI[1:]]
show_job(run_job(LINT_SOFT, cwd=buggy_project), max_lines=3)

ruff позначено `!` (упав, але job продовжився), pytest упав по-справжньому: `test_night` очікував 150, отримав 160. Job червоний через тести. Для лінтера це прийнятний компроміс; для тестів — ні.

### Приклад 2.6. Перевірки справжнього `news_hub`, яким не потрібні залежності

Частину перевірок можна запустити й без встановлення сотні пакетів проєкту: лінтер, компіляція всіх файлів, синтаксис shell-скриптів. Кеш `.pyc` відправляємо в тимчасову папку (`PYTHONPYCACHEPREFIX`), щоб нічого не писати в проєкт.

In [ ]:
NEWS_DIR = ROOT / LESSON / "news_hub"
PYCACHE = tempfile.mkdtemp(prefix="l51_pycache_")

PROJECT_CHECKS = [
    {"name": "ruff check news_hub (пакет застосунку)", "run": "python -m ruff check --no-cache --output-format concise news_hub"},
    {"name": "усі .py компілюються", "run": f"PYTHONPYCACHEPREFIX={PYCACHE} python -m compileall -q news_hub tests migrations && echo ok"},
    {"name": "синтаксис shell-скриптів", "run": "for f in scripts/*.sh; do bash -n \"$f\" && echo \"$f: ok\"; done"},
]

In [ ]:
show_job(run_job(PROJECT_CHECKS, cwd=NEWS_DIR))

Усі три зелені. Повні тести `news_hub` потребують його залежностей (`pip install -r requirements-dev.txt` — FastAPI, SQLAlchemy, fakeredis, aiogram, …); у CI це робить job `tests`. Справжній вивід jobs на PR уроку:

```text
pytest · Python 3.10 / 3.13                 62 с / 56 с   ✅
pytest · мінімальні версії (Python 3.10)    35 с          ✅
pytest і Alembic · PostgreSQL 16 + Redis 7  94 с          ✅
mypy --strict                               42 с          ✅
образ і smoke-тест Compose                  115 с         ✅
публікація образу в ghcr.io                 —             skipped (не main)
```

### Приклад 2.7. Що піде в образ: `requirements.txt` проти `requirements-dev.txt`

Правило уроку 49: в образ — лише залежності роботи; pytest, mypy, fakeredis — у `requirements-dev.txt`, який підключає основний через `-r`.

In [ ]:
def requirement_names(text: str) -> list[str]:
    names = []
    for line in text.splitlines():
        line = line.split("#", 1)[0].strip()
        if line and not line.startswith("-"):
            names.append(re.split(r"[<>=!\[ ;]", line, maxsplit=1)[0].lower())
    return names


DEV_ONLY = {"pytest", "pytest-asyncio", "pytest-cov", "mypy", "fakeredis"}

In [ ]:
runtime = requirement_names((NEWS_DIR / "requirements.txt").read_text(encoding="utf-8"))
dev_text = (NEWS_DIR / "requirements-dev.txt").read_text(encoding="utf-8")
print("в образі:", len(runtime), "пакетів; з інструментів розробки:", sorted(DEV_ONLY & set(runtime)) or "жодного")
print("dev підключає основний:", "-r requirements.txt" in dev_text, "| лише для розробки:", requirement_names(dev_text))

В образі немає жодного інструмента розробки, а `requirements-dev.txt` = основний + тести й типи. Тому в CI `pip install -r requirements-dev.txt`, а в `Dockerfile` — лише `requirements.txt` (книжковий «знайди помилку» в розділі 7 якраз ставить не той файл).

### Приклад 2.8. Мінімальні версії: що зробить `uv --resolution lowest-direct`

`requirements.txt` обіцяє: `djangorestframework-simplejwt>=5.3.1` — «працює з 5.3.1 і новішими». Звичайний CI ставить **найновіші** версії й нижню межу не перевіряє ніколи. `lowest-direct` ставить для кожної **прямої** залежності найменшу дозволену версію. Подивимось, що це означає для проєкту нотаток:

In [ ]:
def lowest_direct(text: str) -> list[str]:
    """name>=X → name==X для кожної прямої залежності (так обирає uv --resolution lowest-direct)."""
    pins = []
    for line in text.splitlines():
        line = line.split("#", 1)[0].strip()
        match = re.match(r"([A-Za-z0-9_.\-\[\]]+)\s*>=\s*([0-9][^,\s]*)", line)
        if match:
            pins.append(f"{match[1]}=={match[2]}")
    return pins

In [ ]:
for pin in lowest_direct((ROOT / LESSON / "crispy_notes_project/requirements.txt").read_text(encoding="utf-8")):
    print(" ", pin)

Саме такий набір перевіряє job `мінімальні версії`. Перший його прогін з межею `>=5.3` упав:

```text
ImportError: Could not import 'rest_framework_simplejwt.authentication.JWTAuthentication' for API setting
'DEFAULT_AUTHENTICATION_CLASSES'. ModuleNotFoundError: No module named 'pkg_resources'.
```

simplejwt 5.3.0 імпортує `pkg_resources` із setuptools, якого в чистому venv немає. Нижню межу піднято до 5.3.1 — і обіцянка в `requirements.txt` знову правдива.

### Приклад 2.9. Перевірки `Dockerfile` без збирання

Кілька правил уроку 49 видно прямо з тексту `Dockerfile`: базовий образ з тегом (не `latest`), залежності окремим шаром **до** коду, користувач не root, `HEALTHCHECK`, `CMD` у exec-формі (JSON-масив — процес стає PID 1 і отримує SIGTERM).

In [ ]:
def dockerfile_checks(text: str) -> dict[str, bool]:
    instructions, buffer = [], ""
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        buffer += line.rstrip("\\") + " "
        if not line.endswith("\\"):
            instructions.append(buffer.strip())
            buffer = ""

    def all_of(keyword: str) -> list[str]:
        return [i for i in instructions if i.split()[0].upper() == keyword]

    position = {i: n for n, i in enumerate(instructions)}
    deps = [i for i in all_of("COPY") if "requirements" in i]
    code_copy = [i for i in all_of("COPY") if i.split()[1] == "."]
    users = all_of("USER")
    return {
        "тег базового образу (не latest)": all(":" in f.split()[1] and not f.split()[1].endswith(":latest") for f in all_of("FROM")),
        "залежності до коду": bool(deps and code_copy) and position[deps[0]] < position[code_copy[0]],
        "USER не root": bool(users) and users[-1].split()[1] not in ("root", "0"),
        "HEALTHCHECK": bool(all_of("HEALTHCHECK")),
        "CMD у exec-формі": bool(all_of("CMD")) and all_of("CMD")[-1][3:].lstrip().startswith("["),
    }


BAD_DOCKERFILE = """
FROM python:latest
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt
CMD uvicorn news_hub.api:app --host 0.0.0.0
"""

In [ ]:
files = {"news_hub": (NEWS_DIR / "Dockerfile").read_text(encoding="utf-8"),
         "нотатки": (ROOT / LESSON / "crispy_notes_project/Dockerfile").read_text(encoding="utf-8"),
         "поганий": BAD_DOCKERFILE}
checks = {name: dockerfile_checks(text) for name, text in files.items()}
for rule in checks["news_hub"]:
    print(f"{rule:<32}" + "".join(f"{name}: {'✓' if result[rule] else '✗'}   " for name, result in checks.items()))
ignore = (NEWS_DIR / ".dockerignore").read_text(encoding="utf-8").splitlines()
print(".env у .dockerignore news_hub:", ".env" in ignore)

Обидва проєкти курсу проходять усі п'ять правил; «поганий» — жодного: `latest` змінюється без попередження, `COPY . .` перед `pip install` скидає кеш шару залежностей на кожну правку коду, root, немає healthcheck, а shell-форма `CMD` запускає `sh -c`, і `docker stop` чекає 10 с, бо SIGTERM отримує shell, а не uvicorn. `.env` в образ не потрапляє.

### Приклад 2.10. Job з Docker (якщо є)

У Colab Docker-демона немає — клітинка друкує, що пропущено. Локально з Docker вона робить те, що job `docker`: **збирає образ** міні-проєкту з базового `python:3.12-slim` (лише якщо він уже є на комп'ютері — нічого не завантажує), **запускає тести в контейнері** і прибирає образ `course-m5-l51-…`. Спершу — помічники.

In [ ]:
import shutil


def docker_ready() -> bool:
    if not shutil.which("docker"):
        return False
    return subprocess.run(["docker", "info"], capture_output=True, text=True).returncode == 0


MINI_DOCKERFILE = """FROM python:3.12-slim
WORKDIR /app
COPY . .
RUN useradd --uid 10001 --no-create-home app
USER app
CMD ["python", "-m", "unittest", "-v"]
"""
IMAGE = "course-m5-l51-mini:ci"

In [ ]:
DOCKER = docker_ready()
base_ready = DOCKER and subprocess.run(["docker", "image", "inspect", "python:3.12-slim"], capture_output=True).returncode == 0
if not DOCKER:
    print("Docker недоступний — пропускаю. Справжній вивід — у блоці нижче.")
elif not base_ready:
    print("Образу python:3.12-slim немає локально — ноутбук нічого не завантажує з мережі. Пропускаю.")
else:
    (clean_project / "Dockerfile").write_text(MINI_DOCKERFILE, encoding="utf-8")
    DOCKER_JOB = [
        {"name": "docker build", "run": f"docker build -q -t {IMAGE} . > /dev/null && echo 'образ зібрано'"},
        {"name": "тести в контейнері", "run": f"docker run --rm {IMAGE}"},
        {"name": "користувач у контейнері", "run": f"docker run --rm {IMAGE} id -un"},
        {"name": "прибрати образ", "if": "always()", "run": f"docker image rm -f {IMAGE} > /dev/null && echo 'образ видалено'"},
    ]
    show_job(run_job(DOCKER_JOB, cwd=clean_project), max_lines=8)

Справжній вивід (з уроку):

```text
✓ docker build: success
      образ зібрано
✓ тести в контейнері: success
      test_day (test_calc.DeliveryPriceTest.test_day) ... ok
      test_night (test_calc.DeliveryPriceTest.test_night) ... ok

      ----------------------------------------------------------------------
      Ran 2 tests in 0.000s

      OK
✓ користувач у контейнері: success
      app
✓ прибрати образ: success
      образ видалено
→ job зелений
```

У справжньому job `docker` курсу після цього йдуть `pytest -m docker` (тести образу з уроку 49) і `scripts/smoke.sh` (увесь стек через nginx, урок 50).

---
## 3. Порядок jobs

Jobs без `needs` стартують одночасно, кожен на своїй машині. `needs` — ребра графа, як `depends_on` у Compose (урок 50): job стартує, коли всі його `needs` завершились **успішно**.

### Приклад 3.1. Хвилі jobs

In [ ]:
def needs_of(job: dict) -> set[str]:
    value = job.get("needs", [])
    return {value} if isinstance(value, str) else set(value)


def job_waves(workflow: dict) -> list[set[str]]:
    needs = {name: needs_of(job) for name, job in workflow["jobs"].items()}
    waves, done = [], set()
    while len(done) < len(needs):
        ready = {name for name, deps in needs.items() if name not in done and deps <= done}
        if not ready:
            raise ValueError("цикл у needs")
        waves.append(ready)
        done |= ready
    return waves

In [ ]:
news_waves = job_waves(NEWS)
print("news_hub:", news_waves)
print("нотатки: ", job_waves(NOTES))
assert news_waves == [{"tests", "min-versions", "postgres", "types"}, {"docker"}, {"publish"}]

Чотири jobs — одразу; `docker` чекає `tests` (нема сенсу збирати образ з кодом, що не пройшов тести); `publish` — усіх.

### Приклад 3.2. Скільки чекати на статус: хвилі проти «по черзі»

Справжні тривалості jobs `news_hub` на PR уроку (секунди). `tests` — matrix з двох jobs, хвиля чекає повільнішого; `publish` на PR пропущено — 0 с.

In [ ]:
DURATIONS = {"tests": [62, 56], "min-versions": [35], "postgres": [94], "types": [42], "docker": [115], "publish": [0]}

In [ ]:
by_waves = sum(max(max(DURATIONS[job]) for job in wave) for wave in news_waves)
serial = sum(sum(times) for times in DURATIONS.values())
print(f"хвилями: {by_waves} с ≈ {by_waves / 60:.1f} хв | одним job по черзі: {serial} с ≈ {serial / 60:.1f} хв")

Паралельність майже вдвічі скорочує очікування: ~3,5 хв проти ~6,7. Але справжній прогін був ще коротшим: старт о 10:52:23, кінець — о 10:55:23, рівно 3 хвилини. Чому?

### Приклад 3.3. Справжній розклад: кожен job чекає лише своїх `needs`

Модель «хвиль» змушує `docker` чекати найповільнішого в першій хвилі (`postgres`, 94 с). GitHub так не робить: `docker` стартує, щойно завершились **його** `needs` — обидва `tests`.

In [ ]:
def schedule(workflow: dict, durations: dict[str, list[int]]) -> dict[str, tuple[int, int]]:
    """{job: (секунда старту, секунда кінця)}: старт — коли завершились усі needs."""
    times: dict[str, tuple[int, int]] = {}
    for wave in job_waves(workflow):
        for name in sorted(wave):
            start = max((times[dep][1] for dep in needs_of(workflow["jobs"][name])), default=0)
            times[name] = (start, start + max(durations[name]))
    return times

In [ ]:
times = schedule(NEWS, DURATIONS)
for name, (start, end) in sorted(times.items(), key=lambda item: item[1]):
    print(f"{name:<13} {start:>4} с → {end:>4} с  {'█' * (max(1, (end - start) // 8)) if end > start else '(skipped)':<16}")
print("статус у PR через", max(end for _, end in times.values()), "с; справжній прогін: 10:52:23 → 10:55:23 = 180 с")

`docker` стартує на 62-й секунді (у справжньому журналі — 10:53:28, через 65 с після старту) і закінчує на 177-й — це й є ~3 хвилини справжнього прогону. Найдовший ланцюжок `tests → docker` — **критичний шлях**: щоб пришвидшити CI, скорочувати треба саме його, а не `postgres`.

### Приклад 3.4. Умова `if:` job `publish`

```yaml
publish:
  if: github.event_name == 'push' && github.ref == 'refs/heads/main'
```

Простий обчислювач для `==`, `!=`, `&&`, `||` над контекстом `github`:

In [ ]:
def eval_if(expression: str, context: dict) -> bool:
    expression = expression.strip()
    if expression.startswith("${{"):
        expression = expression[3:-2]

    def compare(part: str) -> bool:
        left, op, right = re.split(r"\s*(==|!=)\s*", part.strip())
        value, expected = str(lookup(context, left)), right.strip().strip("'")
        return value == expected if op == "==" else value != expected

    return any(all(compare(part) for part in group.split("&&")) for group in expression.split("||"))

In [ ]:
EVENTS = {
    "PR №57":            {"event_name": "pull_request", "ref": "refs/pull/57/merge"},
    "push у гілку PR":   {"event_name": "push", "ref": "refs/heads/claude/m5-production"},
    "push у main":       {"event_name": "push", "ref": "refs/heads/main"},
    "кнопка Run у main": {"event_name": "workflow_dispatch", "ref": "refs/heads/main"},
}
condition = NEWS["jobs"]["publish"]["if"]
for title, github in EVENTS.items():
    print(f"{title:<18} → publish {'запускається' if eval_if(condition, {'github': github}) else 'skipped'}")

Образ публікується лише з `push` у `main` — тобто після злиття PR, коли всі перевірки вже пройшли. Навіть ручний запуск у `main` не публікує: публікація прив'язана до коміту, що потрапив у `main` звичайним шляхом.

### Задача 2. Що буде, коли job упав

Якщо job упав, усі, хто його `needs` (прямо чи через інших), — **skipped**; незалежні jobs працюють далі.

Напиши `job_statuses(workflow, failed)` → словник `{job: "success" | "failure" | "skipped"}`. `failed` — множина jobs, що впали б самі. Ідучи хвилями (`job_waves`): якщо хоч один із `needs_of(job)` не `"success"` — `"skipped"`; інакше, якщо job у `failed` — `"failure"`; інакше `"success"`. Умову `if:` не враховуй.

In [ ]:
def job_statuses(workflow: dict, failed: set[str]) -> dict[str, str]:
    # YOUR CODE HERE
    pass


In [ ]:
result = job_statuses(NEWS, {"tests"})
assert result == {"tests": "failure", "min-versions": "success", "postgres": "success", "types": "success",
                  "docker": "skipped", "publish": "skipped"}, f"упав tests → docker і publish skipped, а вийшло {result}"
assert job_statuses(NEWS, {"types"})["docker"] == "success" and job_statuses(NEWS, {"types"})["publish"] == "skipped"
assert set(job_statuses(NEWS, set()).values()) == {"success"}
assert job_statuses(NOTES, {"deploy-check"}) == {"tests": "success", "min-versions": "success", "deploy-check": "failure",
                                                 "docker": "success", "publish": "skipped"}
print("✅ Задача 2 розв'язана")

### Приклад 3.5. `fail-fast` у matrix

У matrix за замовчуванням `fail-fast: true`: щойно впав один варіант, GitHub **скасовує** решту ще не завершених. Курс ставить `fail-fast: false` — «впала 3.10 — однаково дізнатися, що з 3.13». Модель: 3.10 падає на 20-й секунді, 3.13 був би зеленим на 56-й.

In [ ]:
def matrix_outcome(results: dict[str, tuple[int, bool]], fail_fast: bool) -> dict[str, str]:
    """results: {варіант: (секунда кінця, успіх)} → що побачимо в PR."""
    first_failure = min((end for end, ok in results.values() if not ok), default=None)
    outcome = {}
    for name, (end, ok) in results.items():
        if fail_fast and first_failure is not None and ok and end > first_failure:
            outcome[name] = f"cancelled на {first_failure} с"
        else:
            outcome[name] = f"{'success' if ok else 'failure'} на {end} с"
    return outcome


MATRIX_RUN = {"pytest · Python 3.10": (20, False), "pytest · Python 3.13": (56, True)}

In [ ]:
for fail_fast in (True, False):
    print(f"fail-fast: {str(fail_fast).lower():<5} → {matrix_outcome(MATRIX_RUN, fail_fast)}")

З `fail-fast: true` ти знаєш лише «3.10 упав», а 3.13 скасовано: це баг лише старого Python чи всіх версій? З `false` видно повну картину: проблема тільки в 3.10.

---
## 4. Matrix

### Приклад 4.1. Розгортання

`strategy.matrix` розгортає один job у кілька: **декартів добуток** усіх списків.

In [ ]:
import itertools


def expand(job: dict) -> list[dict]:
    """Комбінації matrix без include/exclude; без matrix — один порожній словник."""
    matrix = job.get("strategy", {}).get("matrix", {})
    keys = [key for key in matrix if key not in ("include", "exclude")]
    return [dict(zip(keys, values)) for values in itertools.product(*(matrix[key] for key in keys))] or [{}]

In [ ]:
print("tests:", expand(NEWS["jobs"]["tests"]), "| types:", expand(NEWS["jobs"]["types"]))
on_pr = sum(len(expand(job)) for name, job in NEWS["jobs"].items() if name != "publish")
print("jobs news_hub на PR:", on_pr)
print("python × db:", expand({"strategy": {"matrix": {"python": ["3.10", "3.13"], "db": ["sqlite", "postgres"]}}}))
assert expand(NEWS["jobs"]["tests"]) == [{"python": "3.10"}, {"python": "3.13"}] and on_pr == 6

На PR `news_hub` запускає 6 jobs (5 описаних, `tests` — подвійний). Дві версії Python × дві бази дали б 4 — кількість росте множенням, тож matrix тримають малою: найстаріша підтримувана й найновіша версія.

### Приклад 4.2. `exclude` і `include`

- `exclude` — прибрати комбінації, що збігаються з усіма ключами запису;
- `include` — до кожної комбінації, де запис **не змінює** вихідних значень matrix, додати його ключі; якщо такої немає — додати запис окремою комбінацією.

In [ ]:
def expand_full(matrix: dict) -> list[dict]:
    keys = [key for key in matrix if key not in ("include", "exclude")]
    combos = [dict(zip(keys, values)) for values in itertools.product(*(matrix[key] for key in keys))] if keys else []
    for rule in matrix.get("exclude", []):
        combos = [c for c in combos if not all(c.get(k) == v for k, v in rule.items())]
    originals = [dict(c) for c in combos]
    for extra in matrix.get("include", []):
        targets = [i for i, original in enumerate(originals)
                   if all(original[k] == v for k, v in extra.items() if k in keys)]
        for i in targets:
            combos[i].update(extra)
        if not targets:
            combos.append(dict(extra))
    return combos


MATRIX = {
    "python": ["3.10", "3.13"],
    "db": ["sqlite", "postgres"],
    "exclude": [{"python": "3.10", "db": "postgres"}],
    "include": [{"python": "3.13", "db": "postgres", "coverage": True},
                {"python": "3.14", "db": "sqlite", "experimental": True}],
}

In [ ]:
for combo in expand_full(MATRIX):
    print(combo)

Із чотирьох комбінацій `exclude` прибрав `3.10 + postgres`. Перший `include` збігся з `3.13 + postgres` і додав йому `coverage` (покриття рахуємо в одному варіанті — пригадай практику книги: `if: matrix.python == '3.13'`). Другий змінює `python` у всіх наявних, тож став окремою комбінацією — так додають експериментальну версію Python.

### Приклад 4.3. Імена jobs у PR

In [ ]:
name_template = "pytest · Python ${{ matrix.python }} · ${{ matrix.db }}"
for combo in expand_full(MATRIX):
    print(" ", render(name_template, {"matrix": combo}))

Кожна комбінація — окремий рядок у списку перевірок PR. Тому в імені job мають бути **всі** змінні matrix: інакше два рядки будуть однакові, і не зрозуміло, який з них червоний.

---
## 5. Кеш

Кожен job — на чистій машині: `pip install` щоразу завантажував би всі пакети. `actions/setup-python` з `cache: pip` зберігає кеш pip між прогонами під **ключем**, у якому є хеш файлів залежностей: змінився `requirements*.txt` — новий ключ, новий кеш.

### Приклад 5.1. `hashFiles`

`hashFiles('…/requirements*.txt')` — SHA-256 від SHA-256 кожного файлу (у сталому порядку): однаковий вміст — однаковий хеш на будь-якій машині.

In [ ]:
import hashlib


def hash_files(paths: list[Path]) -> str:
    total = hashlib.sha256()
    for path in sorted(paths):
        total.update(hashlib.sha256(path.read_bytes()).digest())
    return total.hexdigest()


def pip_cache_key(runner_os: str, python: str, files: list[Path]) -> str:
    """Ключ кешу pip (формат як у setup-python, спрощено)."""
    return f"setup-python-{runner_os}-x64-python-{python}-pip-{hash_files(files)[:16]}"

In [ ]:
requirements = sorted(NEWS_DIR.glob("requirements*.txt"))
print("файли:", [p.name for p in requirements])
for python in ("3.10", "3.13"):
    print(" ", pip_cache_key("Linux", python, requirements))

changed = Path(tempfile.mkdtemp(prefix="l51_req_"))
for path in requirements:
    shutil.copy(path, changed / path.name)
print("той самий вміст:", pip_cache_key("Linux", "3.13", sorted(changed.glob("*.txt"))) == pip_cache_key("Linux", "3.13", requirements))
with open(changed / "requirements.txt", "a", encoding="utf-8") as fh:
    fh.write("rich>=13\n")
print("додали пакет:    ", pip_cache_key("Linux", "3.13", sorted(changed.glob("*.txt"))))

Різні версії Python — різні ключі (пакети з C-розширеннями зібрані під конкретну версію). Копія з тим самим вмістом дає той самий ключ — кеш переживає перезапуск і будь-яку машину; один доданий рядок — зовсім інший хеш.

### Приклад 5.2. Збереження: кеш не перезаписується

Кеш з наявним ключем **не оновлюється**: після job дія `cache` зберігає новий кеш, лише якщо такого ключа ще немає. Тому ключ має змінюватись разом із тим, що в кеші, — інакше кеш назавжди лишиться першою версією.

In [ ]:
def save_cache(key: str, caches: list[dict], now: int) -> str:
    if any(cache["key"] == key for cache in caches):
        return f"{key}: вже є — не зберігаємо"
    caches.append({"key": key, "created": now})
    return f"{key}: збережено"

In [ ]:
store: list[dict] = []
for now, key in enumerate(["pip-Linux-3.13-aaa", "pip-Linux-3.13-aaa", "pip-Linux-3.13-bbb"], 1):
    print(f"прогін {now}: {save_cache(key, store, now)}")
print("у сховищі:", [cache["key"] for cache in store])

Другий прогін з тим самим ключем нічого не зберіг — і правильно: вміст той самий. Третій (змінились залежності) — новий запис.

### Задача 3. Відновлення кешу: точний збіг і `restore-keys`

На початку job `actions/cache` шукає кеш так:

1. запис з **точно таким** ключем → `("hit", ключ)`;
2. інакше — по черзі кожен префікс з `restore_keys`: серед записів, чий ключ **починається** з префікса, береться **найновіший** (`created` найбільший) → `("partial", його ключ)`;
3. нічого не знайшлося → `("miss", None)`.

Часткове відновлення — теж виграш: pip докачає лише те, що змінилось.

Напиши `restore_cache(key, restore_keys, caches)`; `caches` — список словників `{"key": …, "created": …}`.

In [ ]:
def restore_cache(key: str, restore_keys: list[str], caches: list[dict]) -> tuple[str, str | None]:
    # YOUR CODE HERE
    pass


In [ ]:
CACHES = [{"key": "pip-Linux-3.13-aaa", "created": 1},
          {"key": "pip-Linux-3.10-ccc", "created": 2},
          {"key": "pip-Linux-3.13-bbb", "created": 3}]
PREFIXES = ["pip-Linux-3.13-", "pip-Linux-"]
result = restore_cache("pip-Linux-3.13-bbb", PREFIXES, CACHES)
assert result == ("hit", "pip-Linux-3.13-bbb"), f"точний збіг має дати hit, а вийшло {result}"
assert restore_cache("pip-Linux-3.13-zzz", PREFIXES, CACHES) == ("partial", "pip-Linux-3.13-bbb"), "найновіший за першим префіксом"
assert restore_cache("pip-Linux-3.12-zzz", ["pip-Linux-3.12-", "pip-Linux-"], CACHES) == ("partial", "pip-Linux-3.13-bbb")
assert restore_cache("pip-macOS-3.13-x", ["pip-macOS-"], CACHES) == ("miss", None)
assert restore_cache("pip-Linux-3.13-zzz", [], CACHES) == ("miss", None), "без restore-keys — лише точний збіг"
print("✅ Задача 3 розв'язана")

---
## 6. Права і секрети

### Приклад 6.1. Права `GITHUB_TOKEN` кожного job

Кожен прогін отримує тимчасовий `GITHUB_TOKEN`. Його права: блок `permissions` job-а, якщо є; інакше — блок workflow; інакше — налаштування репозиторію (у старших репозиторіях — **запис** усюди).

In [ ]:
def effective_permissions(workflow: dict, job_name: str) -> dict | str:
    job = workflow["jobs"][job_name]
    return job.get("permissions", workflow.get("permissions", "налаштування репозиторію"))

In [ ]:
for job in NEWS["jobs"]:
    print(f"{job:<13} {effective_permissions(NEWS, job)}")
writers = {(wf, job) for wf in ("news_hub", "crispy_notes") for job in WORKFLOWS[wf]["jobs"]
           if isinstance(p := effective_permissions(WORKFLOWS[wf], job), dict) and "write" in p.values()}
print("мають право запису:", writers)
print("без блоку permissions:", effective_permissions(yaml.safe_load("jobs: {test: {}}"), "test"))
assert writers == {("news_hub", "publish"), ("crispy_notes", "publish")}

Писати (у реєстр образів, `packages: write`) може лише `publish` і лише на push у `main`. Решта jobs запускають **код з PR** — їм досить читати. Workflow без `permissions` віддає рішення налаштуванням репозиторію — тому курс завжди пише `permissions: contents: read` зверху.

### Приклад 6.2. `::add-mask::` — секрет не потрапить у журнал

Пароль адміна для smoke генерується в самому прогоні. Будь-яка команда, що його надрукує (помилка curl, `set -x`), показала б його в публічному журналі. Рядок `::add-mask::значення` просить раннер замінювати це значення на `***` у всьому подальшому виводі.

In [ ]:
MASK_JOB = [
    {"name": "згенерувати пароль", "run": 'password="$(python -c "import secrets; print(secrets.token_hex(16))")"\n'
                                          'echo "::add-mask::$password"\n'
                                          'echo "ADMIN_PASSWORD=$password" >> "$GITHUB_ENV"\n'
                                          'echo "пароль: $password"'},
    {"name": "наступний крок бачить змінну", "run": 'echo "довжина: ${#ADMIN_PASSWORD}"; echo "значення: $ADMIN_PASSWORD"'},
    {"name": "без маски (для порівняння)", "run": 'echo "інше значення: $(python -c "print(42 * 1000)")"'},
]

In [ ]:
show_job(run_job(MASK_JOB))

Пароль ніде не надрукувався — лише `***`, хоча крок двічі виводить його явно. Значення без маски друкується як є.

### Приклад 6.3. `$GITHUB_ENV`: змінна для наступних кроків

Кожен крок — окремий процес bash: `export` в одному кроці не видно в наступному. Щоб передати значення далі, крок дописує `ІМ'Я=значення` у файл, шлях до якого дає змінна `$GITHUB_ENV`, — раннер прочитає файл і додасть змінні наступним крокам. У прикладі 6.2 так `ADMIN_PASSWORD` дійшов до другого кроку (довжина 32). Порівняй з `export`:

In [ ]:
EXPORT_JOB = [
    {"name": "export", "run": 'export TOKEN_A=from-export; echo "TOKEN_B=from-github-env" >> "$GITHUB_ENV"'},
    {"name": "що видно далі", "run": 'echo "TOKEN_A=${TOKEN_A:-<порожньо>} TOKEN_B=${TOKEN_B:-<порожньо>}"'},
]

In [ ]:
show_job(run_job(EXPORT_JOB))

`TOKEN_A` з `export` зник разом із процесом першого кроку; `TOKEN_B` з `$GITHUB_ENV` дійшов. Так у job `docker` курсу пароль адміна з кроку генерації `.env` потрапляє в крок `smoke.sh`.

### Приклад 6.4. Секрети в PR з форку

Справжні секрети (ключ LLM, токен бота, SSH-ключ сервера) лежать у **Settings → Secrets and variables → Actions** і читаються як `${{ secrets.ІМ'Я }}`. Але в PR **з форку** GitHub секретів не дає — інакше будь-хто міг би відкрити PR з `echo $SECRET` і забрати ключ. `${{ secrets.X }}` там — порожній рядок, і крок має це пережити:

In [ ]:
LIVE_STEP = 'if [ -z "$GEMINI_API_KEY" ]; then echo "ключа немає — живі тести LLM пропущено (FakeLLM у звичайних тестах)"; ' \
            'else echo "pytest -m llm з ключем ***"; fi'


def live_llm_job(secrets_available: dict) -> list[dict]:
    key = render("${{ secrets.GEMINI_API_KEY }}", {"secrets": secrets_available})
    return run_job([{"name": "живі тести LLM", "run": LIVE_STEP}], env={"GEMINI_API_KEY": key})

In [ ]:
print("PR з форку:");      show_job(live_llm_job({}))
print("push у main:");     show_job(live_llm_job({"GEMINI_API_KEY": "AIza-not-a-real-key"}))

Без ключа крок зелений і чесно каже, що пропустив; з ключем — запускає живі тести. Звичайним тестам курсу ключ не потрібен узагалі (`FakeLLM`, урок 44) — тому CI на PR з форку працює повністю.

Для smoke-тесту курсу секрети **генеруються** в прогоні — `scripts/make_env.py`. Запустимо справжній скрипт проєкту нотаток (лише стандартна бібліотека) на копії в тимчасовій папці й прочитаємо результат, не друкуючи ключ:

In [ ]:
def run_make_env(project: Path) -> tuple[str, dict]:
    copy_dir = Path(tempfile.mkdtemp(prefix="l51_make_env_"))
    (copy_dir / "scripts").mkdir()
    shutil.copy(project / "scripts/make_env.py", copy_dir / "scripts/make_env.py")
    shutil.copy(project / ".env.example", copy_dir / ".env.example")
    status, output, _ = run_step("HTTP_PORT=8080 python scripts/make_env.py", cwd=copy_dir)
    values = dict(line.split("=", 1) for line in (copy_dir / ".env").read_text(encoding="utf-8").splitlines()
                  if "=" in line and not line.startswith("#"))
    return f"код {status}: {output.replace(str(copy_dir), '<тимчасова папка>')}", values

In [ ]:
message, values = run_make_env(ROOT / LESSON / "crispy_notes_project")
print(message)
key = values["DJANGO_SECRET_KEY"]
print("DJANGO_SECRET_KEY: в одинарних лапках:", key.startswith("'") and key.endswith("'"), "| довжина без лапок:", len(key) - 2)
print("HTTP_PORT:", values["HTTP_PORT"], "| POSTGRES_PASSWORD задано:", len(values["POSTGRES_PASSWORD"]) > 10)

Скрипт створив `.env` з випадковим паролем бази й 50-символьним Django-ключем — **в одинарних лапках**, бо в ключі бувають `$` і `#`, а Compose без лапок їх підставив би (урок 50). Порт узято зі змінної, як у кроці CI `HTTP_PORT=8080 python3 scripts/make_env.py`.

---
## 7. Знайди помилку

Workflow для `news_hub` нижче зелений на кожному PR. Проблем у ньому чотири.

### Приклад 7.1. Лінтер workflow

- `"swallowed-exit"` — у `run` є `|| true`;
- `"secret-in-file"` — у `env` (workflow чи job) ім'я з `KEY`, `SECRET`, `TOKEN`, `PASSWORD`, а значення **не** з `${{ secrets.… }}`;
- `"push-everywhere"` — `push` без `branches`;
- `"pr-only-main"` — `pull_request` з `branches`;
- `"dev-deps-missing"` — `pytest` запускається, а встановлено лише `requirements.txt`.

Паролі тестової бази в `services` (`POSTGRES_PASSWORD: news`) лінтер не розглядає: це `env` контейнера-сервісу на одноразовій машині, а не секрет.

In [ ]:
FIND_BUG = yaml.safe_load("""
on:
  push:
  pull_request:
    branches: [main]
jobs:
  test:
    runs-on: ubuntu-latest
    env:
      GEMINI_API_KEY: AIzaSyD-course-demo-key-not-real-0000000
    steps:
      - uses: actions/checkout@v5
      - uses: actions/setup-python@v6
        with: {python-version: "3.13"}
      - run: pip install -r module_5/lessons/lesson_51_ci_cd/news_hub/requirements.txt
      - run: cd module_5/lessons/lesson_51_ci_cd/news_hub && pytest || true
""")
SECRET_WORDS = ("KEY", "SECRET", "TOKEN", "PASSWORD")


def lint(workflow: dict) -> set[str]:
    problems = set()
    rules = triggers(workflow)
    if "push" in rules and "branches" not in rules["push"]:
        problems.add("push-everywhere")
    if "branches" in rules.get("pull_request", {}):
        problems.add("pr-only-main")
    envs = [workflow.get("env", {})] + [job.get("env", {}) for job in workflow["jobs"].values()]
    for env in envs:
        for name, value in env.items():
            if any(word in name.upper() for word in SECRET_WORDS) and "secrets." not in str(value):
                problems.add("secret-in-file")
    for job in workflow["jobs"].values():
        commands = [step.get("run", "") for step in job.get("steps", [])]
        if any("|| true" in command for command in commands):
            problems.add("swallowed-exit")
        installs = " ".join(c for c in commands if "pip install" in c)
        if any("pytest" in c for c in commands) and "requirements.txt" in installs and "requirements-dev" not in installs:
            problems.add("dev-deps-missing")
    return problems

In [ ]:
print("find-bug:", sorted(lint(FIND_BUG)))
print("курс:", {name: sorted(lint(WORKFLOWS[name])) for name in ("news_hub", "crispy_notes")})
assert lint(FIND_BUG) == {"swallowed-exit", "secret-in-file", "push-everywhere", "pr-only-main", "dev-deps-missing"}
assert lint(WORKFLOWS["news_hub"]) == set() and lint(WORKFLOWS["crispy_notes"]) == set()

<details>
<summary>Розбір</summary>

1. `pytest || true` — крок зелений за будь-якого результату (приклад 2.3).
2. Встановлено `requirements.txt` без pytest: `pytest: command not found`, і `|| true` ховає й це. У `news_hub` тести ставлять `requirements-dev.txt` (приклад 2.7).
3. Ключ у файлі — у git назавжди. Секрети — у Settings → Secrets і `${{ secrets.… }}`; тестам ключ не потрібен (`FakeLLM`).
4. `push` без гілок — прогін на кожен push у кожну гілку (разом з PR — двічі на коміт); `pull_request: branches: [main]` — PR в інші гілки без перевірки (приклад 1.3).

Обидва workflows курсу проходять без зауважень. Зверни увагу: `crispy_notes.yml` теж ставить лише `requirements.txt`, але правило `dev-deps-missing` його не чіпає — Django-тести запускає `manage.py test`, а не pytest, і тестові інструменти там не потрібні. Лінтер — підказка, а не суддя: кожне правило пишуть під конкретний проєкт.

`smoke.sh` курсу через `trap cleanup EXIT` теж завжди «прибирає за собою», але код виходу не ковтає: обробник без `exit` не змінює код, з яким завершився скрипт (невдалий вхід адміна → код 1, і job червоний). Прибирання й чесний результат не суперечать одне одному.
</details>

---
## Самоперевірка

1. Чому `pull_request` у курсі — без `branches`? І чим небезпечні «мертві» шаблони `paths`?
2. Що перевіряє job з `uv --resolution lowest-direct`, чого не перевіряє звичайний?
3. Навіщо `needs: tests` у job з образом? Що станеться з `docker` і `publish`, якщо впаде `tests`?
4. Чому справжній прогін коротший за модель «хвиль»?
5. Звідки в job `publish` право `packages: write` і чому не в інших?
6. Що зробить `echo "::add-mask::$password"`, і чому `export` не передає змінну в наступний крок?
7. Чому ключ кешу містить хеш `requirements*.txt`?

<details>
<summary>Відповіді</summary>

1. Інакше PR, націлений не на `main`, залишиться без перевірки, а відсутня перевірка виглядає як успіх (1.3). «Мертвий» шаблон робить те саме тихо: workflow перестає запускатися (1.4).
2. Що нижні межі в `requirements.txt` правдиві: simplejwt 5.3.0 не стартував у чистому venv (2.8).
3. Не збирати образ з кодом, що не пройшов тести, і показати першу причину. Обидва — `skipped` (задача 2).
4. GitHub запускає job, щойно завершились **його** `needs`, а не вся попередня «хвиля»: `docker` не чекає `postgres` (3.3).
5. З блоку `permissions` самого job; решта запускає код з PR і має лише читати (6.1).
6. Замінить значення на `***` у журналі далі (6.2). Кожен крок — окремий процес; міст між кроками — файл `$GITHUB_ENV` (6.3).
7. Змінились залежності — новий ключ і свіжий кеш; кеш з наявним ключем не перезаписується (5.2).
</details>

## Далі

- Книга курсу: [Урок 51](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/lesson_51/) — workflow книги проти курсу, справжні прогони на PR, smoke-тести, публікація образу.
- У репозиторії: `.github/workflows/news_hub.yml`, `crispy_notes.yml`; локально — `pytest`, `mypy --strict news_hub`, `pytest -m docker`, `./scripts/smoke.sh`.
- Модуль 6 — капстоун: власний проєкт з тестами, Docker, Compose і CI з перших днів.